# Kaggle 02 - Verify Qdrant Cloud

Use this notebook to check whether Qdrant Cloud is reachable and whether collections contain points.


In [ ]:
REPO_URL = "https://github.com/phamdinhhai/project-ks2.git"
PROJECT_DIR = "/kaggle/working/project-ks2"

import os
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git pull --ff-only


In [ ]:
!python -m pip install -U pip
!pip install -e ".[qdrant,agent,eval]"
!pip install requests


In [ ]:
# Load Kaggle Secrets. Add these in Notebook > Add-ons > Secrets.
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ["OPENROUTER_API_KEY", "QDRANT_URL", "QDRANT_API_KEY"]:
    try:
        value = secrets.get_secret(name)
    except Exception as exc:
        value = None
        print(f"Secret {name} unavailable: {exc}")
    if value:
        os.environ[name] = value

os.environ.setdefault("OPENROUTER_MODEL", "google/gemini-2.5-flash")
os.environ.setdefault("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
os.environ.setdefault("QDRANT_TEXT_COLLECTION", "text_chunks_prod")
os.environ.setdefault("QDRANT_IMAGE_COLLECTION", "image_patches_prod")
os.environ.setdefault("QDRANT_INDEX_STATE", "/kaggle/working/outputs/index_state/kaggle_index_state.json")

print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))
print("OPENROUTER_MODEL:", os.environ.get("OPENROUTER_MODEL"))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))
print("QDRANT_API_KEY set:", bool(os.environ.get("QDRANT_API_KEY")))
print("Text collection:", os.environ.get("QDRANT_TEXT_COLLECTION"))
print("Image collection:", os.environ.get("QDRANT_IMAGE_COLLECTION"))


In [ ]:
!python scripts/colab_workflow.py test-env
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth


In [ ]:
# Optional local in-memory dry run with mock encoders.
!python -m medical_rag build-qdrant-index --qdrant-url :memory: --limit 20 --use-mock-models
